In [ ]:
# Databricks notebook source
# COMMAND ----------
# %md
# # ETL Process for Insurance Data
# This notebook performs an ETL process on insurance data stored in Unity Catalog tables. It includes data loading, transformation, and writing the final dataset back to Unity Catalog.

# COMMAND ----------
#
import logging
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# COMMAND ----------
#
def load_data():
    """Load data from Unity Catalog tables."""
    demographics_df = spark.table("catalog.insurance_db.demographics")
    policy_df = spark.table("catalog.insurance_db.policy")
    claims_df = spark.table("catalog.insurance_db.claims")
    scores_df = spark.table("catalog.insurance_db.scores")
    aiml_insights_df = spark.table("catalog.insurance_db.aiml_insights")
    return demographics_df, policy_df, claims_df, scores_df, aiml_insights_df

# COMMAND ----------
#
def select_demographics_fields(demographics_df):
    """Select relevant fields from demographics data."""
    return demographics_df.select(
        F.col("Customer_ID"), F.col("Customer_Name"), F.col("Email"), F.col("Phone_Number"), F.col("Address"),
        F.col("City"), F.col("State"), F.col("Postal_Code"), F.col("Date_of_Birth"), F.col("Gender"),
        F.col("Marital_Status"), F.col("Occupation"), F.col("Income_Level"), F.col("Customer_Segment")
    )

# COMMAND ----------
#
def join_data(demographics_selected_df, policy_df, claims_df):
    """Join demographics, policy, and claims data."""
    joined_df = demographics_selected_df.join(
        policy_df, F.col('Customer_ID') == F.col('customer_id'), "inner"
    ).join(
        claims_df, F.col('policy_id') == F.col('Policy_ID'), "inner"
    )
    return joined_df

# COMMAND ----------
#
def aggregate_claims_data(claims_df):
    """Aggregate claims data."""
    return claims_df.groupBy("Customer_ID").agg(
        F.count("Claim_ID").alias("Total_Claims"),
        F.count("Policy_ID").alias("Policy_Count"),
        F.max("Claim_Date").alias("Recent_Claim_Date"),
        F.avg("Claim_Amount").alias("Average_Claim_Amount")
    )

# COMMAND ----------
#
def calculate_metrics(final_df):
    """Calculate additional metrics."""
    claim_to_premium_ratio = F.when(F.col('total_premium_paid') != 0, F.col('Claim_Amount') / F.col('total_premium_paid')).otherwise(0)
    claims_per_policy = F.when(F.col('Policy_Count') != 0, F.col('Total_Claims') / F.col('Policy_Count')).otherwise(0)

    final_df = final_df.withColumn("Age", F.datediff(F.current_date(), F.col('Date_of_Birth')) / 365)
    final_df = final_df.withColumn("Claim_To_Premium_Ratio", claim_to_premium_ratio)
    final_df = final_df.withColumn("Claims_Per_Policy", claims_per_policy)
    final_df = final_df.withColumn("Retention_Rate", F.lit(0.85))
    final_df = final_df.withColumn("Cross_Sell_Opportunities", F.lit("Multi-Policy Discount, Home Coverage Add-on"))
    final_df = final_df.withColumn("Upsell_Potential", F.lit("Premium Vehicle Coverage"))
    return final_df

# COMMAND ----------
#
def select_aiml_insights_fields(aiml_insights_df):
    """Select relevant fields from AI/ML insights data."""
    return aiml_insights_df.select(
        F.col("Customer_ID"), F.col("Churn_Probability"), F.col("Next_Best_Offer"), F.col("Claims_Fraud_Probability"), F.col("Revenue_Potential")
    )

# COMMAND ----------
#
def select_scores_fields(scores_df):
    """Select relevant fields from scores data."""
    return scores_df.select(
        F.col("Customer_ID"), F.col("Credit_Score"), F.col("Fraud_Score"), F.col("Customer_Risk_Score")
    )

# COMMAND ----------
#
def combine_data_sources(final_df, scores_selected_df, aiml_insights_selected_df):
    """Combine all data sources into a comprehensive dataset."""
    return final_df.join(scores_selected_df, "Customer_ID", "inner").join(aiml_insights_selected_df, "Customer_ID", "inner")

# COMMAND ----------
#
def write_final_dataset(customer_360_df):
    """Write the final dataset to Unity Catalog."""
    spark.sql("DROP TABLE IF EXISTS catalog.insurance_db.customer_360")
    customer_360_df.write.format("delta").mode("overwrite").saveAsTable("catalog.insurance_db.customer_360")

# COMMAND ----------
#
def main():
    try:
        # Load data
        demographics_df, policy_df, claims_df, scores_df, aiml_insights_df = load_data()

        # Select relevant fields
        demographics_selected_df = select_demographics_fields(demographics_df)

        # Join data
        joined_df = join_data(demographics_selected_df, policy_df, claims_df)

        # Aggregate claims data
        aggregated_df = aggregate_claims_data(claims_df)

        # Join summarized data with detailed customer data
        final_df = aggregated_df.join(joined_df, "Customer_ID", "inner")

        # Calculate additional metrics
        final_df = calculate_metrics(final_df)

        # Select relevant fields from AI/ML insights and scores data
        aiml_insights_selected_df = select_aiml_insights_fields(aiml_insights_df)
        scores_selected_df = select_scores_fields(scores_df)

        # Combine all data sources
        customer_360_df = combine_data_sources(final_df, scores_selected_df, aiml_insights_selected_df)

        # Write the final dataset
        write_final_dataset(customer_360_df)

        logger.info("ETL process completed successfully.")

    except Exception as e:
        logger.error(f"An error occurred during the ETL process: {e}")
        raise

# COMMAND ----------
#
# Execute the main function
main()
